In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully.")

In [ ]:
df = pd.read_csv('customer_credit_risk_dataset.csv')

print("First 5 rows of the dataset:")
display(df.head())

print(f"\nDataset Shape: {df.shape}")

In [ ]:
print("Dataset Info:")
df.info()

print("\nStatistical Summary:")
display(df.describe())

print("\nMissing Values Count:")
print(df.isnull().sum())

In [ ]:
num_cols = df.select_dtypes(include=['float64', 'int64']).columns
cat_cols = df.select_dtypes(include=['object']).columns

num_imputer = SimpleImputer(strategy='mean')
df[num_cols] = num_imputer.fit_transform(df[num_cols])

cat_imputer = SimpleImputer(strategy='most_frequent')
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

print("Missing values after imputation:")
print(df.isnull().sum().sum())

In [ ]:
def cap_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    df[column] = np.where(df[column] < lower_bound, lower_bound, df[column])
    df[column] = np.where(df[column] > upper_bound, upper_bound, df[column])
    return df

outlier_cols = ['annual_income', 'loan_amount', 'credit_score', 'spending_ratio']

for col in outlier_cols:
    df = cap_outliers(df, col)

print("Outliers handled successfully.")

In [ ]:
df['join_date'] = pd.to_datetime(df['join_date'])

df['join_year'] = df['join_date'].dt.year
df['join_month'] = df['join_date'].dt.month
df['join_day'] = df['join_date'].dt.day

df = df.drop('join_date', axis=1)

print("Date features extracted.")
display(df[['join_year', 'join_month', 'join_day']].head())

In [ ]:
le = LabelEncoder()
df['gender'] = le.fit_transform(df['gender'])
df['education_level'] = le.fit_transform(df['education_level'])

df = pd.get_dummies(df, columns=['region', 'employment_type', 'loan_purpose'], drop_first=True)

print("Encoding completed. New shape:", df.shape)
display(df.head())

In [ ]:
scale_cols = ['age', 'annual_income', 'loan_amount', 'credit_score', 
              'repayment_history', 'transaction_count', 'spending_ratio']

scaler = StandardScaler()

df[scale_cols] = scaler.fit_transform(df[scale_cols])

print("Scaling applied. Check the first row values:")
display(df[scale_cols].head())

In [ ]:
df['debt_to_income'] = df['loan_amount'] / (df['annual_income'] + 1)

df['avg_monthly_trans'] = df['transaction_count'] / 6

print("New features created:")
display(df[['debt_to_income', 'avg_monthly_trans']].head())

In [ ]:
print("Final Dataset Shape:", df.shape)

plt.figure(figsize=(10, 6))
sns.heatmap(df.corr()[['default_flag']].sort_values(by='default_flag', ascending=False), 
            annot=True, cmap='coolwarm')
plt.title("Correlation with Default Flag")
plt.show()

df.to_csv('cleaned_credit_risk_data.csv', index=False)
print("Cleaned dataset saved as 'cleaned_credit_risk_data.csv'")